# RQ3 — Zero-Shot Signer Generalization + Domain-Adversarial Training (DANN)

Separate from `🧩LuckBasedModel(2).ipynb` (RQ1: naive vs group-aware split leakage audit; RQ2: single vs stacked LSTM — both done, out of scope here, do not port their variable names into this notebook).

This notebook:
- Holds out **อัยด้า** entirely (zero clips in any training condition) as a true zero-shot test signer, per the Whisper (Radford et al. 2022) evaluation philosophy the paper borrows.
- Trains 3 stacked-LSTM models — `solo-baseline` (กร), `pooled-baseline` (ตะวัน+ปิ่น), `pooled+DANN` (ตะวัน+ปิ่น with a gradient-reversal signer-invariance objective) — and evaluates all three on the identical, untouched อัยด้า test set.
- Adds a risk-coverage / selective-accuracy analysis, formalizing the confidence-threshold rejection already shipped in `scripts/predict_stream.py`.

Requires `data/processed/` and `data/augmented/` (for ตะวัน, ปิ่น, อัยด้า) synced to this same Drive path — they were extracted locally and need uploading before this notebook can see them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import tensorflow as tf
print('TF version:', tf.__version__)

In [ ]:
BASE = "/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/data"

# 10-word conversational vocabulary only — finish/none are shared, non-signer-specific
# utility classes with no signer label, and are excluded from this experiment.
# Deliberately named TEN_WORDS, not TARGET_SIGNS, to avoid colliding with the
# RQ1/RQ2 notebook's 12-class constant if these ever end up in the same session.
TEN_WORDS = [
    "คนหูหนวก", "คุณ", "ช่วย", "ขอบคุณ", "ฉัน",
    "ต้องการ", "เข้าใจ", "ไม่", "ถาม", "บอก",
]

POOLED_SIGNERS = ["ตะวัน", "ปิ่น"]
SOLO_SIGNER = "กร"
ZEROSHOT_SIGNER = "อัยด้า"

# กร_ถาม_40.npy is an orphaned processed sample with no matching raw video
# (raw only has 39 ถาม clips) — real, loadable data, but excluded from solo-train
# subsampling so it isn't silently included/excluded by chance.
KNOWN_ORPHAN = "กร_ถาม_40.npy"

SPLIT_SEED = 123  # distinct from augment_keypoints.py's seed=42

In [ ]:
def parse_filename(stem):
    """'กร_คุณ_3' -> ('กร', 'คุณ', '3'). Returns None for finish/none
    (no signer prefix) or any unexpected filename."""
    for sign in TEN_WORDS:
        marker = f"_{sign}_"
        if marker in stem:
            signer, _, num = stem.partition(marker)
            return signer, sign, num
    return None


def list_originals_by_signer(sign, signer):
    d = os.path.join(BASE, "processed", sign)
    if not os.path.exists(d):
        return []
    out = []
    for f in os.listdir(d):
        if not f.endswith(".npy"):
            continue
        parsed = parse_filename(f[:-4])
        if parsed and parsed[0] == signer:
            out.append(os.path.join(d, f))
    return out


def list_augments_for_original(sign, orig_stem):
    """orig_stem e.g. 'กร_คุณ_3' -> matches 'กร_คุณ_3_aug_0'..'_aug_4'."""
    d = os.path.join(BASE, "augmented", sign)
    if not os.path.exists(d):
        return []
    return [
        os.path.join(d, f) for f in os.listdir(d)
        if f.endswith(".npy") and f[:-4].rsplit("_aug_", 1)[0] == orig_stem
    ]

In [ ]:
def build_zeroshot_test():
    """อัยด้า, originals only, never augmented. 10 signs x 10 clips = 100 samples."""
    X, y, meta = [], [], []
    for label_idx, sign in enumerate(TEN_WORDS):
        for p in list_originals_by_signer(sign, ZEROSHOT_SIGNER):
            X.append(np.load(p)); y.append(label_idx); meta.append(os.path.basename(p))
    return np.array(X), np.array(y), meta


def build_pooled_train():
    """ตะวัน + ปิ่น, all originals + all their augments. No internal split needed —
    the disjoint test set is an entirely separate signer (อัยด้า), not held-out
    videos, so there is no augmentation-leakage risk in this arm."""
    X, y, meta, signer_y = [], [], [], []
    for label_idx, sign in enumerate(TEN_WORDS):
        for signer_idx, signer in enumerate(POOLED_SIGNERS):
            for p in list_originals_by_signer(sign, signer):
                stem = os.path.basename(p)[:-4]
                X.append(np.load(p)); y.append(label_idx); meta.append(stem); signer_y.append(signer_idx)
                for aug_p in list_augments_for_original(sign, stem):
                    X.append(np.load(aug_p)); y.append(label_idx)
                    meta.append(os.path.basename(aug_p)[:-4]); signer_y.append(signer_idx)
    return np.array(X), np.array(y), np.array(signer_y), meta


def build_solo_train(n_per_word=20, seed=SPLIT_SEED):
    """กร, subsampled to n_per_word originals/word (matching the pooled arm's
    scale), then augmented. Known orphan sample explicitly excluded from the
    candidate pool before subsampling."""
    rng = np.random.default_rng(seed)
    X, y, meta = [], [], []
    for label_idx, sign in enumerate(TEN_WORDS):
        origs = sorted(list_originals_by_signer(sign, SOLO_SIGNER))
        origs = [p for p in origs if os.path.basename(p) != KNOWN_ORPHAN]
        n_take = min(n_per_word, len(origs))
        chosen_idx = rng.choice(len(origs), size=n_take, replace=False)
        for i in chosen_idx:
            p = origs[i]
            stem = os.path.basename(p)[:-4]
            X.append(np.load(p)); y.append(label_idx); meta.append(stem)
            for aug_p in list_augments_for_original(sign, stem):
                X.append(np.load(aug_p)); y.append(label_idx); meta.append(os.path.basename(aug_p)[:-4])
    return np.array(X), np.array(y), meta

In [ ]:
# --- Sanity check before training anything: exact counts + disjointness ---
X_zeroshot, y_zeroshot, meta_zeroshot = build_zeroshot_test()
X_pooled, y_pooled, signer_pooled, meta_pooled = build_pooled_train()
X_solo, y_solo, meta_solo = build_solo_train()

print(f"zero-shot test (อัยด้า):  X={X_zeroshot.shape}  y={y_zeroshot.shape}  (expect 100 total, 10/word)")
print(f"pooled-train (ตะวัน+ปิ่น): X={X_pooled.shape}  y={y_pooled.shape}  signer_y={signer_pooled.shape}")
print(f"solo-train (กร, 20/word): X={X_solo.shape}  y={y_solo.shape}")

# disjointness: no signer in the zero-shot set should ever appear in either training arm
zs_signers = {parse_filename(m[:-4] if m.endswith('.npy') else m)[0] for m in meta_zeroshot}
assert zs_signers == {ZEROSHOT_SIGNER}, f"unexpected signers in zero-shot set: {zs_signers}"
print(f"\nDisjointness OK: zero-shot test set contains only {zs_signers}")

In [ ]:
from sklearn.model_selection import train_test_split

# Internal validation split of the pooled-train data, for early-stopping monitoring
# ONLY — never used as a reported result, and completely separate from the
# อัยด้า zero-shot test set. Stratified by sign so both classes appear in both halves.
(X_pooled_train, X_pooled_val,
 y_pooled_train, y_pooled_val,
 signer_pooled_train, signer_pooled_val) = train_test_split(
    X_pooled, y_pooled, signer_pooled,
    test_size=0.15, random_state=SPLIT_SEED, stratify=y_pooled,
)

# Same internal val split for the solo arm (sign-stratified only — single signer, no signer label needed)
X_solo_train, X_solo_val, y_solo_train, y_solo_val = train_test_split(
    X_solo, y_solo, test_size=0.15, random_state=SPLIT_SEED, stratify=y_solo,
)

print(f"pooled: train={X_pooled_train.shape} val={X_pooled_val.shape}")
print(f"solo:   train={X_solo_train.shape} val={X_solo_val.shape}")

## Models

`build_baseline_model()` — plain stacked LSTM, identical shape to the RQ2 winner, no auxiliary branch at all (used for both `solo-baseline` and `pooled-baseline`).

`build_dann_model()` — same primary path, plus a Gradient-Reversal-Layer auxiliary signer classifier branching off the shared 32-d LSTM feature vector. The auxiliary head is training-only; only the primary path's 4 weighted layers would ever be exported for deployment.

In [ ]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense, Layer
from tensorflow.keras.callbacks import EarlyStopping, Callback

N_SIGNS = len(TEN_WORDS)
N_POOLED_SIGNERS = len(POOLED_SIGNERS)


def build_baseline_model():
    inp = Input(shape=(30, 126), name="keypoints")
    x = LSTM(32, return_sequences=True, name="lstm1")(inp)
    x = Dropout(0.5, name="dropout1")(x)
    x = LSTM(32, name="lstm2")(x)
    x = Dropout(0.5, name="dropout2")(x)
    h = Dense(16, activation="relu", name="dense1")(x)
    out = Dense(N_SIGNS, activation="softmax", name="sign_output")(h)
    model = Model(inputs=inp, outputs=out, name="baseline")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

In [ ]:
@tf.custom_gradient
def _grl_op(x, lamb):
    def grad(dy):
        return -lamb * dy, tf.zeros_like(lamb)
    return tf.identity(x), grad


class GradientReversalLayer(Layer):
    """Forward pass = identity. Backward pass = multiply incoming gradient by
    -lambda. lambda is a mutable tf.Variable so GRLLambdaSchedule can anneal it
    across epochs without rebuilding the model.

    IMPORTANT: self.lamb is converted to a plain tensor via tf.convert_to_tensor
    before entering _grl_op. Passing the tf.Variable itself into a
    tf.custom_gradient function makes TF's GradientTape treat it as a "watched
    variable", which then requires the inner grad() function to declare a
    `variables=` keyword argument -- and without it TF raises:
    "ValueError: @tf.custom_gradient grad_fn must accept keyword arguments for
    each GradientTape watched variable." Reading the variable into a tensor
    first avoids that entirely."""

    def __init__(self, lamb=0.0, **kwargs):
        super().__init__(**kwargs)
        self.lamb = tf.Variable(lamb, trainable=False, dtype=tf.float32, name="grl_lambda")

    def call(self, x):
        lamb_tensor = tf.convert_to_tensor(self.lamb)
        return _grl_op(x, lamb_tensor)


def build_dann_model(grl_lambda_init=0.0):
    inp = Input(shape=(30, 126), name="keypoints")
    x = LSTM(32, return_sequences=True, name="lstm1")(inp)
    x = Dropout(0.5, name="dropout1")(x)
    x = LSTM(32, name="lstm2")(x)
    features = Dropout(0.5, name="dropout2")(x)

    # primary head — same shape/order as build_baseline_model's, so its 4
    # weighted layers could be copied into a plain Sequential for export.
    h = Dense(16, activation="relu", name="dense1")(features)
    sign_output = Dense(N_SIGNS, activation="softmax", name="sign_output")(h)

    # auxiliary head — training-only, discarded before any export
    grl = GradientReversalLayer(lamb=grl_lambda_init, name="grl")
    reversed_features = grl(features)
    a = Dense(16, activation="relu", name="adv_dense1")(reversed_features)
    signer_output = Dense(N_POOLED_SIGNERS, activation="softmax", name="signer_output")(a)

    model = Model(inputs=inp, outputs=[sign_output, signer_output], name="dann")
    model.compile(
        optimizer="adam",
        loss={"sign_output": "sparse_categorical_crossentropy",
              "signer_output": "sparse_categorical_crossentropy"},
        loss_weights={"sign_output": 1.0, "signer_output": 1.0},
        metrics={"sign_output": "accuracy", "signer_output": "accuracy"},
    )
    return model, grl


class GRLLambdaSchedule(Callback):
    """2-stage schedule: lambda=0 for warmup_epochs (let the feature extractor
    learn the primary task first), then jump to target_lambda. Simpler than the
    full DANN sigmoid ramp; if training is unstable, lower target_lambda
    (e.g. 0.5 -> 0.2) before abandoning DANN."""

    def __init__(self, grl_layer, warmup_epochs=10, target_lambda=0.5):
        super().__init__()
        self.grl_layer = grl_layer
        self.warmup_epochs = warmup_epochs
        self.target_lambda = target_lambda

    def on_epoch_begin(self, epoch, logs=None):
        new_val = 0.0 if epoch < self.warmup_epochs else self.target_lambda
        self.grl_layer.lamb.assign(new_val)

## Run 1 — solo-baseline (กร, 20/word)

In [ ]:
solo_model = build_baseline_model()
solo_history = solo_model.fit(
    X_solo_train, y_solo_train,
    validation_data=(X_solo_val, y_solo_val),
    epochs=200, batch_size=16,
    callbacks=[EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)],
)

## Run 2 — pooled-baseline (ตะวัน+ปิ่น)

In [ ]:
pooled_model = build_baseline_model()
pooled_history = pooled_model.fit(
    X_pooled_train, y_pooled_train,
    validation_data=(X_pooled_val, y_pooled_val),
    epochs=200, batch_size=16,
    callbacks=[EarlyStopping(monitor="val_loss", patience=20, restore_best_weights=True)],
)

## Run 3 — pooled+DANN (ตะวัน+ปิ่น, signer-invariance objective)

In [ ]:
dann_model, grl_layer = build_dann_model()
dann_history = dann_model.fit(
    X_pooled_train, {"sign_output": y_pooled_train, "signer_output": signer_pooled_train},
    validation_data=(X_pooled_val, {"sign_output": y_pooled_val, "signer_output": signer_pooled_val}),
    epochs=200, batch_size=16,
    callbacks=[
        EarlyStopping(monitor="val_sign_output_loss", mode="min", patience=20, restore_best_weights=True),
        GRLLambdaSchedule(grl_layer, warmup_epochs=10, target_lambda=0.5),
    ],
)
print("history keys:", list(dann_history.history.keys()))  # confirm exact metric names before plotting below

## Zero-shot accuracy comparison (the headline result)

In [ ]:
def zeroshot_predict_probs(model, X, is_dann=False):
    preds = model.predict(X, verbose=0)
    return preds[0] if is_dann else preds


results = {}
for name, model, is_dann in [
    ("solo-baseline", solo_model, False),
    ("pooled-baseline", pooled_model, False),
    ("pooled+DANN", dann_model, True),
]:
    probs = zeroshot_predict_probs(model, X_zeroshot, is_dann=is_dann)
    preds = probs.argmax(axis=1)
    acc = (preds == y_zeroshot).mean()
    results[name] = {"probs": probs, "preds": preds, "accuracy": acc}
    print(f"{name:18s} zero-shot accuracy on อัยด้า: {acc:.4f}")

print("\nPer-class breakdown (small N=10/class — report per-class, not just aggregate):")
for name, r in results.items():
    print(f"\n{name}:")
    for label_idx, sign in enumerate(TEN_WORDS):
        mask = y_zeroshot == label_idx
        class_acc = (r["preds"][mask] == y_zeroshot[mask]).mean()
        print(f"  {sign:10s}: {class_acc:.2f}  (n={mask.sum()})")

## DANN invariance sanity check (required — this is the evidence for the DANN claim)

Expected pattern: `signer_output_accuracy` climbs during the lambda=0 warmup (confirms the signer signal is learnable at all), then degrades back toward chance (50%, 2 pooled signers) once lambda activates. If it stays high post-warmup, invariance was not achieved — treat that as a stop-and-fix signal, not a result to report as-is.

In [ ]:
import matplotlib.pyplot as plt

h = dann_history.history
plt.figure(figsize=(8, 5))
plt.plot(h["signer_output_accuracy"], label="Train signer-acc (adversarial)")
plt.plot(h["val_signer_output_accuracy"], label="Val signer-acc (adversarial)")
plt.axhline(0.5, color="gray", linestyle="--", label="Chance level (2 signers)")
plt.axvline(10, color="red", linestyle=":", label="GRL lambda activated (epoch 10)")
plt.xlabel("Epoch"); plt.ylabel("Signer-classifier accuracy")
plt.title("DANN invariance sanity check")
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/dann_invariance_check.png", dpi=150)
plt.show()

# also sanity-check the primary task didn't crater relative to pooled-baseline
plt.figure(figsize=(8, 5))
plt.plot(pooled_history.history["val_accuracy"], label="pooled-baseline val accuracy")
plt.plot(h["val_sign_output_accuracy"], label="pooled+DANN val sign accuracy")
plt.xlabel("Epoch"); plt.ylabel("Sign-classification accuracy")
plt.title("Primary-task accuracy: baseline vs DANN (should track closely)")
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

## Risk-coverage / selective-accuracy analysis

Formalizes the confidence-threshold rejection already shipped in `scripts/predict_stream.py` into a reported metric on the zero-shot อัยด้า test set. Caveat: only 100 test samples — high-threshold/low-coverage regions get statistically noisy fast (annotate with n_covered, focus narrative on the 40-100% coverage range).

In [ ]:
def compute_risk_coverage(probs, y_true, n_points=100):
    """Sweep coverage directly (most-confident-first), rather than a fixed
    absolute confidence threshold. A fixed threshold makes each model's curve
    span a different coverage range depending on how confident it happens to
    be (a poorly-calibrated model can end up with a much narrower, misleadingly
    good-looking domain) -- sweeping by coverage percentile instead guarantees
    every model's curve spans the full 0-1 range, making AURC comparable
    across models."""
    confidences = probs.max(axis=1)
    preds = probs.argmax(axis=1)
    correct = (preds == y_true).astype(float)

    order = np.argsort(-confidences)  # most confident first
    correct_sorted = correct[order]

    n = len(y_true)
    coverages = np.linspace(1.0 / n, 1.0, n_points)
    accuracies, n_covered = [], []
    for cov in coverages:
        k = max(1, int(round(cov * n)))
        accuracies.append(correct_sorted[:k].mean())
        n_covered.append(k)
    return coverages, np.array(accuracies), np.array(n_covered)


def aurc(coverages, accuracies):
    risks = 1 - accuracies
    order = np.argsort(coverages)
    return np.trapz(risks[order], coverages[order])


plt.figure(figsize=(8, 6))
for name, r in results.items():
    cov, acc, n_cov = compute_risk_coverage(r["probs"], y_zeroshot)
    area = aurc(cov, acc)
    plt.plot(cov, acc, label=f"{name} (AURC={area:.4f})")
    # annotate a couple of reference coverage points with their sample count
    for target_cov in [0.5, 0.25]:
        idx = np.argmin(np.abs(cov - target_cov))
        plt.annotate(f"n={n_cov[idx]}", (cov[idx], acc[idx]), fontsize=7)

plt.xlabel("Coverage (fraction of predictions accepted)")
plt.ylabel("Selective accuracy (accuracy among accepted predictions)")
plt.title("Risk-coverage curve — zero-shot อัยด้า test set")
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/KachornLuckbasedThSL/KachornThSL/risk_coverage_curve.png", dpi=150)
plt.show()

## Notes for write-up (do not skip when reporting results)

- **Small-N**: 100 zero-shot test samples (10/class), ~20-21/word pooled-training. Report per-class breakdowns, not just aggregate accuracy — a single flipped prediction moves a class's accuracy by 10 points.
- **Only 2 pooled signers**: the auxiliary classifier is a binary, not multi-way, signer discriminator — a narrower invariance claim than typical DANN papers with many source domains. State this limitation directly.
- **Augmented signer-label correlation**: augmented copies inherit their original's signer identity trivially (geometric transforms don't change who signed), so the signer classifier sees 6x correlated views per real clip — not a leakage problem, but can make the invariance-degradation curve look more dramatic than on originals alone.
- **Deployment scope**: this experiment is 10-way (no finish/none); the shipped model is 12-way. This notebook's artifacts are research checkpoints for the paper, not a drop-in replacement for the deployed model.